In [ ]:
!pip uninstall -y langchain langchain-community langchain-openai langchain-experimental langchain-chroma langchainhub -q
!pip uninstall -y sentence-transformers unstructured python-dotenv streamlit -q

In [ ]:
!pip install -q \
langchain==0.3.25 \
langchain-community==0.3.23 \
langchain-openai==0.3.11 \
python-dotenv==1.0.1 \
streamlit==1.44.0 \
langchain-experimental==0.3.4 \
sentence-transformers==3.4.1 \
langchain-chroma==0.2.4 \
langchainhub==0.1.21 \
unstructured==0.17.2

In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader

In [ ]:
urls=['https://www.rlbschools.org/who-we-are/','https://www.rlbschools.org/infrastructure/school-building/', 'https://www.rlbschools.org/affiliation/', 'https://www.rlbschools.org/rules-regulations/']
loader=UnstructuredURLLoader(urls=urls)
data= loader.load()

In [ ]:
len(data)

4

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

#split data
text_splitter= RecursiveCharacterTextSplitter(chunk_size=1000)
docs= text_splitter.split_documents(data)

In [ ]:
print("Total number of documents:", len(docs))

Total number of documents: 13


In [ ]:
docs[0]

Document(metadata={'source': 'https://www.rlbschools.org/who-we-are/'}, page_content='About us : An Overview')

In [ ]:
docs[1]

Document(metadata={'source': 'https://www.rlbschools.org/who-we-are/'}, page_content='Education as planned during British period had been primarily a government activity, supplemented by aided schools financed by generous donors, and Public Schools run by the elites exclusively for their children. A few decades after independence, academic revolution took place in the form of self financed schools for masses. At this juncture Sri Jai Pal Singh, a philanthropist, patriot and educationist, conceived of schools founded on the ideals of patriotism and humanism, imparting comprehensive education to masses at very moderate fees which the middle class and lower middle class parents could easily afford. His dream born in 1970 with one school has grown up as a vast edifice of 10 schools functioning in their own permanent buildings. Six of them are affiliated to CBSE Delhi, one to CISCE, New Delhi and other three are recognized by Department of Education, U.P. Government. The special characteris

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI

In [26]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')


'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'

In [29]:
import os

In [31]:
os.environ["OPENAI_API_KEY"]= userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_BASE"] = userdata.get('BASE_URL')

In [33]:
vectorstore= Chroma.from_documents(documents=docs, embedding=OpenAIEmbeddings())

In [34]:
retriever= vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [35]:
retrieved_docs= retriever.invoke("what is  schedule for admission for LKG class?")

In [38]:
print(retrieved_docs[0].page_content)

RULES & REGULATIONS

ADMISSION

Schedule For LKG Class, admission starts in the last week of January. Admission for other classes upto XI, depending on available vacancy, starts in the last week of March Eligibilty

To be eligible for admission to LKG class, the candidate should be three & half year and above.

For admission to other classes, the candidate should have passed the previous class from a recognised school.

Procedure

Candidate will apply for admission on the prescribed form along with the photocopy of passed Report Card of previous class.

On the basis of test and interview candidates will be shortlisted.

For proof of date of birth, they will produce certificates signed by the competent authority/ Nagar Nigam or the transfer certificates from recognised schools.

The selected candidates are required to bring their two latest passport size coloured photographs & One family photograph (father & mother only) of size (3.5 c.m. X 4.5 c.m.) at the time of Admission.


In [46]:
llm= OpenAI(temperature=0.4, max_tokens= 500)

In [41]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of context to answer the question at the end of the question. "
    "If you don't know the answer, say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [42]:
question_ans_chain= create_stuff_documents_chain(llm , prompt)
rag_chain= create_retrieval_chain(retriever, question_ans_chain)

In [45]:
response= rag_chain.invoke({"input": "what is  schedule for admission for LKG class?"})
print(response["answer"])

BadRequestError: Error code: 400 - {'error': {'message': 'Input required: specify "prompt"', 'code': 400}, 'user_id': 'user_3BFGFdlM1HQHnaL6D9mO5krgLHm'}